<a href="https://colab.research.google.com/github/itsmeyessir/llm-decon/blob/main/llm-eval-harness/retrieval_context_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Runtime Environment & Background Service Initialization

In a Retrieval-Augmented Generation (RAG) system, isolating the retrieval evaluation framework requires a deterministic, local LLM-as-a-Judge runtime.

### Architectural Considerations
* **Daemon Isolation:** We deploy the Ollama runtime binary directly to the host environment. To prevent blocking the synchronous Jupyter event loop, the service daemon is initialized asynchronously via `subprocess.Popen`.
* **Health Polling:** Rather than relying on non-deterministic delays (`time.sleep`), the execution loop actively polls the HTTP socket endpoint (`http://127.0.0.1:11434/`) until an active `HTTP 200` state is returned, guaranteeing downstream cells execute only on verified daemon readiness.

In [6]:
# -------------------------------------------------------------------
# System Dependency Injection & Daemon Initialization
# -------------------------------------------------------------------

# Provision OS-level dependencies for binary extraction (quiet mode)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd

# Fetch and execute the Ollama deployment script via standard streams
!curl -fsSL https://ollama.com/install.sh | sh

# Provision Python runtime packages
!pip install -q ragas langchain-community langchain-ollama langchain-huggingface pandas

import subprocess
import time
import urllib.request
import urllib.error

def initialize_ollama_daemon():
    """
    Forks the Ollama server process from the main Jupyter kernel execution thread.
    Utilizes subprocess.Popen to prevent blocking the synchronous notebook runtime.
    """
    print("[SYSTEM] Initializing Ollama daemon process...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

def wait_for_socket(url="http://127.0.0.1:11434/", timeout=30, poll_interval=0.5):
    """
    Actively polls the HTTP daemon endpoint until a verified HTTP 200 response
    is received, guaranteeing downstream cells only run on verified server readiness.
    """
    print(f"[SYSTEM] Polling daemon socket at {url}...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    elapsed = round(time.time() - start_time, 2)
                    print(f"[SYSTEM] Verified socket connection (HTTP 200) in {elapsed}s.")
                    print("[SYSTEM] Runtime environment fully provisioned and operational.")
                    return True
        except (urllib.error.URLError, ConnectionRefusedError, OSError):
            time.sleep(poll_interval)

    raise RuntimeError(f"[CRITICAL] Daemon failed to respond on {url} within {timeout} seconds.")

# Execution Pipeline
initialize_ollama_daemon()
wait_for_socket()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
[SYSTEM] Initializing Ollama daemon process...
[SYSTEM] Polling daemon socket at http://127.0.0.1:11434/...
[SYSTEM] Verified socket connection (HTTP 200) in 0.0s.
[SYSTEM] Runtime environment fully provisioned and operational.


True

## 2. Model Provisioning & Manifest Registration

Evaluating retrieval performance requires both a reasoning engine to act as the evaluator judge and a dense vector embedding model to compute semantic alignment.

### Target Provisioning
* **Evaluator Model:** `Llama 3.1 8B Instruct` (4-bit quantized GGUF). Serves as the structured judge for context ranking and factual extraction.
* **Embedding Model:** `sentence-transformers/all-MiniLM-L6-v2`. Used for semantic vector similarity calculations across context chunks.
* **Idempotency Control:** The provisioning workflow queries the local REST API (`/api/tags`) prior to dispatching pull requests, preventing redundant multi-gigabyte network transfers on re-execution.

In [7]:
# -------------------------------------------------------------------
# Model Provisioning, Idempotency & Manifest Registration
# -------------------------------------------------------------------
import json
import subprocess
import urllib.request
import urllib.error

def get_registered_models(daemon_url="http://127.0.0.1:11434/api/tags"):
    """
    Queries the local Ollama REST API to retrieve the current inventory of pulled models.
    Returns a dictionary mapping model names to their size in gigabytes.
    """
    try:
        req = urllib.request.Request(daemon_url)
        with urllib.request.urlopen(req, timeout=5) as response:
            if response.status == 200:
                payload = json.loads(response.read().decode("utf-8"))
                models = payload.get("models", [])
                inventory = {}
                for m in models:
                    name = m.get("name", "")
                    size_gb = round(m.get("size", 0) / (1024**3), 2)
                    inventory[name] = size_gb
                return inventory
    except Exception as e:
        print(f"[WARN] Failed to query daemon inventory: {str(e)}")
        return {}


def ensure_model_provisioned(target_model="llama3.1:8b"):
    """
    Idempotent model provisioner:
    1. Checks if model exists locally.
    2. Pulls model if absent, capturing process exit codes.
    3. Verifies post-pull registration and reports exact system state.
    """
    print(f"[SYSTEM] Inspecting local inventory for model '{target_model}'...")
    inventory = get_registered_models()

    matched_name = next((name for name in inventory if target_model in name or name in target_model), None)

    if matched_name:
        size = inventory[matched_name]
        print(f"[SYSTEM] Local inventory hit: Model '{matched_name}' ({size} GB) is ready. Skipping download.")
        return True

    print(f"[SYSTEM] Model '{target_model}' not found in local inventory. Initiating pull sequence...")

    try:
        process = subprocess.run(
            ["ollama", "pull", target_model],
            check=True,
            text=True
        )

        updated_inventory = get_registered_models()
        verified_match = next((name for name in updated_inventory if target_model in name or name in target_model), None)

        if verified_match:
            size = updated_inventory[verified_match]
            print(f"[SYSTEM] Provisioning successful: Model '{verified_match}' ({size} GB) verified in manifest.")
            return True
        else:
            raise RuntimeError(f"Pull command completed, but '{target_model}' is missing from daemon manifest.")

    except subprocess.CalledProcessError as e:
        print(f"[CRITICAL] Model pull process failed with exit code {e.returncode}.")
        raise RuntimeError(f"Failed to pull model '{target_model}'. Check network connection or model tag.") from e
    except Exception as e:
        print(f"[CRITICAL] Provisioning pipeline encountered an error: {str(e)}")
        raise e


# 1. Ensure Evaluator LLM is provisioned
ensure_model_provisioned("llama3.1:8b")

# 2. Ensure Embedding Engine is pre-cached
print("[SYSTEM] Pre-fetching local semantic embedding engine (all-MiniLM-L6-v2)...")
try:
    from langchain_huggingface import HuggingFaceEmbeddings
    # Instantiating it here triggers the Hugging Face hub download into the local cache
    # so it does not block the evaluation loop in Cell 4.
    _ = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("[SYSTEM] Embedding model cached and verified.")
except Exception as e:
    print(f"[CRITICAL] Failed to provision embedding model: {str(e)}")

[SYSTEM] Inspecting local inventory for model 'llama3.1:8b'...
[SYSTEM] Local inventory hit: Model 'llama3.1:8b' (4.58 GB) is ready. Skipping download.
[SYSTEM] Pre-fetching local semantic embedding engine (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[SYSTEM] Embedding model cached and verified.


## 3. Retrieval Failure Mode Matrix & Data Schema

To calibrate the retrieval evaluation harness independently from the generator LLM, we construct a curated vector dataset representing four distinct database retrieval failure states.

### Simulated Failure Modes
1. **Baseline Control (Optimal Fetch):** The retrieved context contains only the relevant factual statements required to answer the prompt.
2. **Low Precision / High Noise (Needle in a Haystack):** The vector database retrieves the correct factual chunk, but surrounds it with multiple irrelevant paragraphs. Tests if the metric catches database noise.
3. **Low Recall / Partial Fetch (Information Deficit):** The prompt requires multi-part factual synthesis, but the database only returns a subset of the required chunks.
4. **Complete Retrieval Failure (Irrelevant Noise):** The vector database returns context chunks with zero semantic overlap with the prompt or ground truth.

In [8]:
# -------------------------------------------------------------------
# Retrieval Failure Mode Matrix Construction & Schema Validation
# -------------------------------------------------------------------
import pandas as pd
from datasets import Dataset

def construct_and_validate_retrieval_dataset(eval_data: dict) -> Dataset:
    """
    Constructs a Hugging Face Dataset from raw retrieval test vectors while
    enforcing strict schema integrity, dimensional alignment, and non-null constraints.
    """
    required_keys = {"question", "contexts", "ground_truth", "answer"}

    print("[SYSTEM] Initiating retrieval evaluation dataset schema validation...")

    # 1. Structural Schema Check
    missing_keys = required_keys - set(eval_data.keys())
    if missing_keys:
        raise ValueError(f"[CRITICAL] Dataset schema missing required keys: {missing_keys}")

    # 2. Dimensional Alignment Check across vectors
    lengths = {key: len(eval_data[key]) for key in required_keys}
    unique_lengths = set(lengths.values())

    if len(unique_lengths) > 1:
        raise ValueError(f"[CRITICAL] Dimensional mismatch across dataset columns: {lengths}")

    record_count = list(unique_lengths)[0]
    print(f"[SYSTEM] Dimensional check passed: {record_count} evaluation scenarios detected.")

    # 3. Validation of Types & Non-Null Elements
    for idx in range(record_count):
        if not isinstance(eval_data["question"][idx], str) or not eval_data["question"][idx].strip():
            raise TypeError(f"[CRITICAL] Invalid or empty 'question' at index {idx}.")

        if not isinstance(eval_data["contexts"][idx], list) or not eval_data["contexts"][idx]:
            raise TypeError(f"[CRITICAL] 'contexts' at index {idx} must be a non-empty list of string chunks.")

        if not isinstance(eval_data["ground_truth"][idx], str) or not eval_data["ground_truth"][idx].strip():
            raise TypeError(f"[CRITICAL] Invalid or empty 'ground_truth' at index {idx}.")

        if not isinstance(eval_data["answer"][idx], str) or not eval_data["answer"][idx].strip():
            raise TypeError(f"[CRITICAL] Invalid or empty 'answer' at index {idx}.")

    print("[SYSTEM] Structural schema and data integrity verified.")
    dataset = Dataset.from_dict(eval_data)
    print(f"[SYSTEM] Hugging Face Dataset instantiated ({len(dataset)} total records).")
    return dataset

# -------------------------------------------------------------------
# Curated Vector DB Retrieval Test Matrix
# -------------------------------------------------------------------
raw_retrieval_data = {
    "question": [
        # Scenario 1: Optimal Fetch (Control)
        "What is the required initial response time for a Severity 1 service outage?",

        # Scenario 2: Low Precision / High Noise (Needle in a Haystack)
        "What are the system RAM requirements for running the local inference engine?",

        # Scenario 3: Low Recall / Partial Fetch (Information Deficit)
        "What are the two mandatory authentication factors required for administrative portal access?",

        # Scenario 4: Complete Retrieval Failure (Irrelevant Noise)
        "What is the maximum allowed file upload size for user profile avatars?"
    ],

    "contexts": [
        # Scenario 1: Only relevant facts retrieved
        [
            "Enterprise Service Level Agreement (SLA): Severity 1 (Critical Outage) requires an initial response within 15 minutes."
        ],

        # Scenario 2: Relevant fact retrieved alongside multiple irrelevant noisy chunks
        [
            "The web portal frontend requires 2GB of client-side RAM for optimal rendering in standard browsers.",
            "Hardware Specification: The local inference engine requires a minimum of 16GB unified system RAM.",
            "Database backups run automatically every midnight UTC and consume up to 500MB of temporary disk buffer.",
            "Network bandwidth should be at least 100Mbps for smooth streaming operations across local instances."
        ],

        # Scenario 3: Only 1 out of 2 necessary facts retrieved by vector search
        [
            "Security Policy Section 4.1: Administrative access requires Hardware Security Key (FIDO2) verification."
            # Missing Chunk: TOTP requirement omitted to simulate partial retrieval failure
        ],

        # Scenario 4: Irrelevant chunks returned due to poor vector embedding match
        [
            "Profile avatars must be in PNG or JPEG format.",
            "User accounts are automatically locked after five consecutive failed login attempts.",
            "Password renewal is enforced every 90 calendar days across all user tiers."
        ]
    ],

    "ground_truth": [
        "Severity 1 outages require an initial response within 15 minutes.",
        "The local inference engine requires a minimum of 16GB unified system RAM.",
        "Administrative portal access requires both a Hardware Security Key (FIDO2) and a Time-based One-Time Password (TOTP).",
        "The maximum allowed file upload size for profile avatars is 5MB."
    ],

    "answer": [
        "The initial response time for a Severity 1 outage is 15 minutes.",
        "The local inference engine requires at least 16GB of system RAM.",
        "Administrative portal access requires a Hardware Security Key (FIDO2).",
        "Profile avatars must be in PNG or JPEG format."
    ]
}

# Instantiate and validate dataset
eval_dataset = construct_and_validate_retrieval_dataset(raw_retrieval_data)

# Summary table preview
df_summary = eval_dataset.to_pandas()
print("\n[SYSTEM] Retrieval Evaluation Vector Summary:")
for idx, row in df_summary.iterrows():
    print(f"\n--- Scenario {idx + 1} ---")
    print(f"Question:        {row['question']}")
    print(f"Context Chunks:  {len(row['contexts'])} retrieved chunk(s)")
    print(f"Ground Truth:    {row['ground_truth']}")

[SYSTEM] Initiating retrieval evaluation dataset schema validation...
[SYSTEM] Dimensional check passed: 4 evaluation scenarios detected.
[SYSTEM] Structural schema and data integrity verified.
[SYSTEM] Hugging Face Dataset instantiated (4 total records).

[SYSTEM] Retrieval Evaluation Vector Summary:

--- Scenario 1 ---
Question:        What is the required initial response time for a Severity 1 service outage?
Context Chunks:  1 retrieved chunk(s)
Ground Truth:    Severity 1 outages require an initial response within 15 minutes.

--- Scenario 2 ---
Question:        What are the system RAM requirements for running the local inference engine?
Context Chunks:  4 retrieved chunk(s)
Ground Truth:    The local inference engine requires a minimum of 16GB unified system RAM.

--- Scenario 3 ---
Question:        What are the two mandatory authentication factors required for administrative portal access?
Context Chunks:  1 retrieved chunk(s)
Ground Truth:    Administrative portal access requir

## 4. Context Metric Isolation & Execution Pipeline

We isolate the database retrieval evaluation by stripping out candidate generation metrics (`faithfulness`, `answer_relevancy`) and focusing exclusively on context metrics.

### Metric Definitions
* **Context Precision:** Measures the signal-to-noise ratio of the retrieved chunks. Calculates whether relevant chunks are ranked at the top of the context payload rather than buried under irrelevant text.

$$\text{Context Precision} = \frac{\text{Relevant Chunks in Top } K}{\text{Total Retrieved Chunks in } K}$$

* **Context Recall:** Measures the factual completeness of the retrieved payload. Evaluates if every sentence in the `ground_truth` can be attributed back to the retrieved `contexts`.

$$\text{Context Recall} = \frac{|\text{Ground Truth Sentences Attributable to Context}|}{|\text{Total Ground Truth Sentences}|}$$

In [9]:
# -------------------------------------------------------------------
# Metric Evaluation, AST Sanitization & Context Isolation
# -------------------------------------------------------------------
import warnings
import re
import sys
import time
from types import ModuleType

# Suppress deprecation warnings from upstream framework instability
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# Architecture Patch: Dependency Decoupling
# -----------------------------------------------------------
# Ragas internally maintains legacy hardcoded imports. To maintain
# a 100% local footprint without Google Cloud PIP dependencies,
# we inject a phantom module into sys.modules.
print("[SYSTEM] Deploying Phantom Module Patch for Ragas dependency bypass...")
DummyClass = type("DummyClass", (object,), {})

dummy_chat = ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = DummyClass
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

try:
    import langchain_community.llms
    langchain_community.llms.VertexAI = DummyClass
except ImportError:
    dummy_llms = ModuleType("langchain_community.llms")
    dummy_llms.VertexAI = DummyClass
    sys.modules["langchain_community.llms"] = dummy_llms

# -----------------------------------------------------------
# Core Imports
# -----------------------------------------------------------
from ragas import evaluate
from ragas.metrics import context_precision, context_recall
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# -----------------------------------------------------------
# Middleware: Output Sanitization
# -----------------------------------------------------------
class JSONSanitizingOllamaWrapper(ChatOllama):
    """
    Middleware interceptor for LLM outputs. Small parameter models (like 8B)
    occasionally prepend conversational text or leave trailing commas in JSON responses.
    This wrapper extracts the valid JSON buffer before passing it to Ragas AST parsers.
    """
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        result = super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)
        for generation in result.generations:
            if hasattr(generation, 'message'):
                content = generation.message.content

                start_idx = content.find('{')
                end_idx = content.rfind('}')

                if start_idx != -1 and end_idx != -1:
                    clean_json = content[start_idx : end_idx + 1]
                    # Scrub invalid trailing commas in arrays/objects
                    clean_json = re.sub(r',\s*(?=["}\]])', '', clean_json)
                    generation.message.content = clean_json
        return result

# -----------------------------------------------------------
# Component Initialization & Evaluation Harness Execution
# -----------------------------------------------------------
print("[SYSTEM] Initializing Llama 3.1 (8B) Judge for Context Evaluation...")
base_llm = JSONSanitizingOllamaWrapper(
    model="llama3.1:8b",
    temperature=0,
    format="json",
    system="Respond with raw JSON only. No explanations. Ensure strict schema adherence."
)
evaluator_llm = LangchainLLMWrapper(base_llm)

print("[SYSTEM] Loading local semantic embedding engine (all-MiniLM-L6-v2)...")
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

print("\n[SYSTEM] Commencing Ragas Context Evaluation Harness...")
print("[SYSTEM] Processing retrieval vectors across Context Precision and Context Recall...")

start_time = time.time()

try:
    results = evaluate(
        dataset=eval_dataset,
        metrics=[context_precision, context_recall],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings
    )

    elapsed_time = round(time.time() - start_time, 2)
    print(f"\n[SYSTEM] Retrieval context matrix computed successfully in {elapsed_time}s.")

except Exception as e:
    print(f"\n[CRITICAL] Context evaluation harness execution failed: {str(e)}")

[SYSTEM] Deploying Phantom Module Patch for Ragas dependency bypass...
[SYSTEM] Initializing Llama 3.1 (8B) Judge for Context Evaluation...
[SYSTEM] Loading local semantic embedding engine (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[SYSTEM] Commencing Ragas Context Evaluation Harness...
[SYSTEM] Processing retrieval vectors across Context Precision and Context Recall...


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]


[SYSTEM] Retrieval context matrix computed successfully in 52.51s.


## 5. Automated Retrieval Scorecard & Bottleneck Isolation

We compute the retrieval evaluation matrix using Ragas and map the results into a structured telemetry scorecard.

### Diagnostic Objectives
* **Isolating Database vs. LLM Failures:** High generator accuracy cannot fix a low `context_recall` score. This scorecard mathematically identifies whether a RAG failure originated in the vector indexing/search layer or the downstream synthesis layer.
* **Schema Alignment:** AST sanitization middleware enforces valid JSON parsing from the judge model, preventing parser crashes during multi-chunk context evaluations.

In [10]:
# -------------------------------------------------------------------
# Telemetry Formatting & Root Cause Analysis Scorecard
# -------------------------------------------------------------------
import pandas as pd

try:
    # 1. Extract evaluation results into a Pandas DataFrame
    df = results.to_pandas()

    # 2. Round metric columns for a cleaner UI
    metric_cols = ['context_precision', 'context_recall']
    df[metric_cols] = df[metric_cols].round(4)

    # 3. Format context chunks (Prevents massive text blobs from ruining the table UI)
    if 'retrieved_contexts' in df.columns:
        df['chunk_count'] = df['retrieved_contexts'].apply(lambda x: f"{len(x)} chunk(s)")
    else:
        # Fallback for older package versions
        df['chunk_count'] = df['contexts'].apply(lambda x: f"{len(x)} chunk(s)")

    # 4. Map columns to Ragas v0.2+ internal schema
    display_cols = ['user_input', 'chunk_count', 'context_precision', 'context_recall']

    # 5. Render Scorecard
    print("\n================ RETRIEVAL EVALUATION SCORECARD ================\n")
    display(df[display_cols])
    print("\n================================================================\n")

    # 6. Automated Diagnostic Interpretation
    print("[SYSTEM] Automated Root Cause Diagnostics:")
    print("-> Scenario 1 (Control)      : Expected High Precision, High Recall (Database fetched perfectly).")
    print("-> Scenario 2 (Haystack)     : Expected Low Precision, High Recall (Database over-fetched noise).")
    print("-> Scenario 3 (Partial)      : Expected High/Med Precision, Low Recall (Database missed critical information).")
    print("-> Scenario 4 (Irrelevant)   : Expected Zero Precision, Zero Recall (Database fetched completely wrong chunks).")

except Exception as e:
    print(f"\n[CRITICAL] Failed to generate scorecard telemetry: {str(e)}")


================ RETRIEVAL EVALUATION SCORECARD ================



,user_input,chunk_count,context_precision,context_recall
0,What is the required initial response time for...,1 chunk(s),1.0,1.0
1,What are the system RAM requirements for runni...,4 chunk(s),0.5,0.4
2,What are the two mandatory authentication fact...,1 chunk(s),1.0,1.0
3,What is the maximum allowed file upload size f...,3 chunk(s),0.0,0.0




[SYSTEM] Automated Root Cause Diagnostics:
-> Scenario 1 (Control)      : Expected High Precision, High Recall (Database fetched perfectly).
-> Scenario 2 (Haystack)     : Expected Low Precision, High Recall (Database over-fetched noise).
-> Scenario 3 (Partial)      : Expected High/Med Precision, Low Recall (Database missed critical information).
-> Scenario 4 (Irrelevant)   : Expected Zero Precision, Zero Recall (Database fetched completely wrong chunks).


## 6. Telemetry Analysis & Architectural Takeaways

The automated retrieval scorecard successfully evaluated the simulated vector database, but more importantly, it exposed the mechanical limits of using a small 8B-parameter model as a judge.

### Telemetry Breakdown
* **The Extremes (Indices 0 & 3):** The 8B judge performs flawlessly on clear-cut boundaries. It correctly awarded perfect `1.0` scores for an optimal fetch and correctly bottomed out at `0.0` when the vector search returned completely irrelevant noise.
* **The Noise Penalty (Index 1):** The database retrieved the correct fact but buried it in irrelevant chunks. The judge correctly dropped `context_precision` to `0.5000`. However, the noise confused the 8B model's reasoning capabilities, causing it to incorrectly penalize `context_recall` (0.4000) even though the necessary fact was present.
* **The Leniency Flaw (Index 2):** The ground truth required two distinct facts (FIDO2 and TOTP), but the database only provided one. A robust judge should score recall at `0.5000`. The 8B model pattern-matched the first requirement, stopped reasoning, and falsely awarded a perfect `1.0000`.

### Production Systems Conclusion
Local 8B models provide a highly efficient, zero-cost first pass for catching blatant vector search failures and hallucinations. However, they lack the sustained attention mechanism (unless fine-tuned for that specific task) required for complex, multi-variable factual synthesis.

In a production CI/CD pipeline, an optimal architecture would route basic `context_precision` checks to local lightweight models to save costs, while escalating `context_recall` validations to a 70B+ parameter model (or cloud API) to ensure strict logical validation.